In [ ]:
import kagglehub

# Download latest version
path_r = kagglehub.dataset_download("guy007/nutrientdeficiencysymptomsinrice")

print("Path to dataset files:", path_r)

100%|██████████| 989M/989M [00:07<00:00, 135MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/guy007/nutrientdeficiencysymptomsinrice/versions/1


In [ ]:
path_m = kagglehub.dataset_download("ashishpatelresearch/maize-plant-leaf-nutrient-deficiency-dataset")

print("Path to dataset files:", path_m)

100%|██████████| 343M/343M [00:02<00:00, 167MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ashishpatelresearch/maize-plant-leaf-nutrient-deficiency-dataset/versions/1


In [ ]:
import os
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# --- CONFIGURATION ---
# Change this to your actual folder path
base_path = 'drive/MyDrive/nutrient-deficiency/rice_1'
target_total = 1000

# Subfolders to process
folders = ['Nitrogen(N)', 'Phosphorus(P)', 'Potassium(K)']

# --- AUGMENTATION SETTINGS ---
# Keeping settings moderate to preserve nutrient deficiency features
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='reflect'
)

for folder_name in folders:
    folder_path = os.path.join(base_path, folder_name)

    if not os.path.exists(folder_path):
        print(f"Skipping {folder_name}: Folder not found.")
        continue

    # Get only the original images (ignore existing augmented ones if you run this twice)
    current_images = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    num_current = len(current_images)

    if num_current >= target_total:
        print(f"✅ {folder_name} already has {num_current} images.")
        continue

    num_needed = target_total - num_current
    print(f"🔄 {folder_name}: Found {num_current}, generating {num_needed} more...")

    generated_count = 0
    while generated_count < num_needed:
        # Pick a random original image to augment
        img_name = random.choice(current_images)
        img_path = os.path.join(folder_path, img_name)

        try:
            img = load_img(img_path)
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)

            # Flow generates 1 image and saves it back to the SAME folder
            for batch in datagen.flow(x, batch_size=1,
                                      save_to_dir=folder_path,
                                      save_prefix='aug',
                                      save_format='jpg'):
                generated_count += 1
                break
        except Exception as e:
            print(f"Could not process {img_name}: {e}")
            continue

print("\n✨ All folders updated to 1000 images.")

🔄 Nitrogen(N): Found 440, generating 560 more...
🔄 Phosphorus(P): Found 333, generating 667 more...
🔄 Potassium(K): Found 383, generating 617 more...

✨ All folders updated to 1000 images.


In [ ]:
import os
import pandas as pd

def find_csv_file(directory):
    """Recursively finds the first CSV file in a directory."""
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith('.csv'):
                return os.path.join(root, file)
    return None

# Try to find and load the CSV for path_r
csv_path_r = find_csv_file(path_r)
if csv_path_r:
    print(f"Loading data from: {csv_path_r}")
    df1 = pd.read_csv(csv_path_r)
    print("Head of df1:")
    display(df1.head())
else:
    print(f"No CSV file found in {path_r}. Please specify which file to load.")
    df1 = None

# Try to find and load the CSV for path_m
csv_path_m = find_csv_file(path_m)
if csv_path_m:
    print(f"\nLoading data from: {csv_path_m}")
    df2 = pd.read_csv(csv_path_m)
    print("Head of df2:")
    display(df2.head())
else:
    print(f"\nNo CSV file found in {path_m}. Please specify which file to load.")
    df2 = None


No CSV file found in /root/.cache/kagglehub/datasets/guy007/nutrientdeficiencysymptomsinrice/versions/1. Please specify which file to load.

No CSV file found in /root/.cache/kagglehub/datasets/ashishpatelresearch/maize-plant-leaf-nutrient-deficiency-dataset/versions/1. Please specify which file to load.


In [ ]:
# center crop for coffee images belonging to flowering stage

import cv2
import os
from pathlib import Path

def smart_crop_to_leaf(input_folder, output_folder, target_size=224):
    """
    Takes 'Tree' images and crops the center to get 'Leaf-level' detail.
    """
    path = Path(input_folder)
    out_path = Path(output_folder)
    out_path.mkdir(parents=True, exist_ok=True)

    for img_file in path.glob("*.jpg"): # or *.png
        img = cv2.imread(str(img_file))
        if img is None: continue

        h, w, _ = img.shape
        # Calculate center coordinates
        center_x, center_y = w // 2, h // 2

        # We want to take a square that is roughly 40-50% of the tree size
        # to ensure we get a cluster of leaves/berries
        crop_width = int(min(h, w) * 0.5)

        start_x = max(0, center_x - crop_width // 2)
        start_y = max(0, center_y - crop_width // 2)

        # Slicing the image (Crop)
        cropped = img[start_y:start_y+crop_width, start_x:start_x+crop_width]

        # Resize to your final model input size
        final_img = cv2.resize(cropped, (target_size, target_size))

        cv2.imwrite(str(out_path / img_file.name), final_img)

# Example usage for your Stage 3 Coffee Tree images
smart_crop_to_leaf('Coffee/3_Flowering_Trees', 'Coffee/3_Flowering_Leaves')
print("Cropping complete! Your coffee images now match the scale of Rice/Maize.")

Cropping complete! Your coffee images now match the scale of Rice/Maize.


In [ ]:
# data augmentation for coffee seedling stage images
import albumentations as A
import cv2
import os
import random
from pathlib import Path

# --- CONFIGURATION ---
INPUT_DIR = "Coffee/Stage1_Raw"  # Folder with your few seedling images
OUTPUT_DIR = "Coffee/1"          # Folder where 1000 images will go
TOTAL_IMAGES_NEEDED = 1000

# Define the "Variations" (Tilts, Flips, Brightness)
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.3),
])

os.makedirs(OUTPUT_DIR, exist_ok=True)
images = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Generating images... target: {TOTAL_IMAGES_NEEDED}")

for i in range(TOTAL_IMAGES_NEEDED):
    # Pick a random image from your small set
    image_name = random.choice(images)
    image = cv2.imread(os.path.join(INPUT_DIR, image_name))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Apply the magic
    augmented = transform(image=image)["image"]

    # Save the new version
    save_path = os.path.join(OUTPUT_DIR, f"seedling_aug_{i}.jpg")
    cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

print(f"Done! Check {OUTPUT_DIR} for your 1,000 images.")

In [ ]:
import os
import cv2
import random
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# --- CONFIGURATION ---
# Set this to your specific Coffee P folder path
folder_path = 'drive/MyDrive/nutrient-deficiency/coffee_deficiency/phosphorus-P'
target_total = 1000
save_prefix = 'aug_p'

# 1. Define Augmentation Strategy
# We use 'reflect' to keep the leaf edges natural for Phosphorus identification
datagen = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='reflect'
)

# 2. Identify current images
image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
num_current = len(image_files)

if num_current >= target_total:
    print(f"✅ Folder already has {num_current} images. No action taken.")
else:
    num_needed = target_total - num_current
    print(f"🚀 Current: {num_current} | Goal: {target_total} | Generating: {num_needed}")

    generated_count = 0

    # Loop through original images to create variations
    while generated_count < num_needed:
        for filename in image_files:
            if generated_count >= num_needed:
                break

            img_path = os.path.join(folder_path, filename)

            try:
                # Load image
                img = load_img(img_path)
                x = img_to_array(img)
                x = x.reshape((1,) + x.shape) # Reshape for the generator

                # Generate and Save
                # flow() saves directly to the folder_path
                for batch in datagen.flow(x, batch_size=1,
                                          save_to_dir=folder_path,
                                          save_prefix=save_prefix,
                                          save_format='jpg'):
                    generated_count += 1
                    break # Move to the next seed image
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                continue

    print(f"✨ Success! {folder_path} now contains {len(os.listdir(folder_path))} total images.")

🚀 Current: 354 | Goal: 1000 | Generating: 646
✨ Success! drive/MyDrive/nutrient-deficiency/coffee_deficiency/phosphorus-P now contains 983 total images.


In [ ]:
import os

# --- CONFIGURATION ---
# Replace with your folder path (e.g., 'coffee_p_deficiency')
folder_path = 'drive/MyDrive/nutrient-deficiency/coffee_deficiency/phosphorus-P'

def count_images(path):
    # List of common image extensions
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

    if not os.path.exists(path):
        return "Folder not found."

    # Count files that end with the valid extensions
    files = [f for f in os.listdir(path) if f.lower().endswith(valid_extensions)]
    return len(files)

# Execute
image_count = count_images(folder_path)
print(f"📊 Total images in folder: {image_count}")

📊 Total images in folder: 977


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import drive
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

# --- SETTINGS ---
BASE_PATH = '/content/drive/MyDrive/nutrient-deficiency'
IMG_SIZE = 224
BATCH_SIZE = 32

crop_map = {'rice_deficiency': 0, 'maize_deficiency': 1, 'coffee_deficiency': 2}
label_map = {'Nitrogen(N)': 0, 'Phosphorus(P)': 1, 'Potassium(K)': 2, 'Healthy': 3}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Create the Master List
data = []
for crop_folder, crop_id in crop_map.items():
    crop_path = os.path.join(BASE_PATH, crop_folder)
    for label_name, label_id in label_map.items():
        f_path = os.path.join(crop_path, label_name)
        if os.path.exists(f_path):
            for img in os.listdir(f_path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    data.append([os.path.join(f_path, img), crop_id, label_id])

df = pd.DataFrame(data, columns=['filepath', 'crop_id', 'label_id'])
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=42)

# Training Dataset Pipeline
train_ds = tf.data.Dataset.from_tensor_slices((train_df['filepath'], train_df['crop_id'], train_df['label_id']))
train_ds = train_ds.map(lambda f, c, l: (
    (tf.image.resize(tf.image.decode_image(tf.io.read_file(f), channels=3, expand_animations=False), [IMG_SIZE, IMG_SIZE]) / 255.0, tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
), num_parallel_calls=tf.data.AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Validation Dataset Pipeline
val_ds = tf.data.Dataset.from_tensor_slices((val_df['filepath'], val_df['crop_id'], val_df['label_id']))
val_ds = val_ds.map(lambda f, c, l: (
    (tf.image.resize(tf.image.decode_image(tf.io.read_file(f), channels=3, expand_animations=False), [IMG_SIZE, IMG_SIZE]) / 255.0, tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
import numpy as np
from PIL import Image

# --- 1. THE SAFE GENERATOR (Captures corrupt files & keeps training alive) ---
def safe_generator(df_partition):
    for _, row in df_partition.iterrows():
        try:
            # Attempt to open and process using PIL for safety
            img = Image.open(row['filepath']).convert('RGB')
            img = img.resize((IMG_SIZE, IMG_SIZE))
            img_array = np.array(img) / 255.0

            # Metadata Input (One-hot for Rice, Maize, Coffee)
            crop_oh = np.zeros(3)
            crop_oh[row['crop_id']] = 1

            # Target Output (One-hot for N, P, K, Healthy)
            label_oh = np.zeros(4)
            label_oh[row['label_id']] = 1

            yield (img_array, crop_oh), label_oh

        except Exception:
            # This handles the "Uncompress failed" error by skipping the file
            print(f" SKIPPING CORRUPT FILE: {row['filepath']}")
            continue

# --- 2. CONVERT TO TF DATASETS ---
sig = ((tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(3,), dtype=tf.float32)),
       tf.TensorSpec(shape=(4,), dtype=tf.float32))

train_ds = tf.data.Dataset.from_generator(lambda: safe_generator(train_df), output_signature=sig).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_generator(lambda: safe_generator(val_df), output_signature=sig).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- 3. ARCHITECTURE ---
img_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
crop_in = layers.Input(shape=(3,), name="crop_input")

# EfficientNetV2S Backbone
base = tf.keras.applications.EfficientNetV2S(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

x = layers.GlobalAveragePooling2D()(base(img_in))
x = layers.Dropout(0.3)(x)

# Fusion Logic
y = layers.Dense(16, activation='relu')(crop_in)
merged = layers.Concatenate()([x, y])
z = layers.Dense(256, activation='relu')(merged)
z = layers.Dropout(0.4)(z)
out = layers.Dense(4, activation='softmax')(z)

model = models.Model(inputs=[img_in, crop_in], outputs=out)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# --- 4. EXECUTION ---
print(" Stage 1: Warm-up (10 Epochs)")
model.fit(train_ds, validation_data=val_ds, epochs=10)

print(" Stage 2: Fine-Tuning (20 Epochs)")
# Note: In the Functional API, the base model is usually model.layers[2]
model.layers[2].trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=20)

model.save('deficiency_power_model.keras')
print(" Done! Your power model is saved.")

 Stage 1: Warm-up (10 Epochs)
Epoch 1/10
